In [1]:
import pandas as pd
import re
import random
import ast
from sklearn.model_selection import train_test_split


In [ ]:
# load dataset from hugging face
from datasets import load_dataset
ds = load_dataset("arbml/ashaar")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/126M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/151M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/254630 [00:00<?, ? examples/s]

In [3]:
df = ds["train"].to_pandas()
df.columns

Index(['poem title', 'poem meter', 'poem verses', 'poem theme', 'poem url',
       'poet name', 'poet description', 'poet url', 'poet era',
       'poet location', 'poem description', 'poem language type'],
      dtype='object')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 254630 entries, 0 to 254629
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   poem title          162865 non-null  object
 1   poem meter          153353 non-null  object
 2   poem verses         254630 non-null  object
 3   poem theme          67520 non-null   object
 4   poem url            254592 non-null  object
 5   poet name           254630 non-null  object
 6   poet description    69911 non-null   object
 7   poet url            162694 non-null  object
 8   poet era            147421 non-null  object
 9   poet location       64028 non-null   object
 10  poem description    14470 non-null   object
 11  poem language type  183407 non-null  object
dtypes: object(12)
memory usage: 23.3+ MB


In [5]:
df.head()

,poem title,poem meter,poem verses,poem theme,poem url,poet name,poet description,poet url,poet era,poet location,poem description,poem language type
0,أصبح الملك للذي فطر الخلق,بحر الخفيف,"[أَصبَحَ المُلك لِلَّذي فَطر الخَل, قَ بِتَقدي...",قصيدة دينية,https://www.aldiwan.net/poem16182.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,None,None,None
1,من أي مولى ارتجي,بحر مجزوء الرمل,"[مِن أَي مَولى اِرتَجي, وَلاي باب التَجي, وَال...",قصيدة دينية,https://www.aldiwan.net/poem16183.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,None,None,None
2,العبد عبدك يا من أنت سيده,بحر البسيط,"[العَبد عَبدك يا مَن أَنتَ سَيدهُ, وَلَيسَ غَي...",قصيدة ذم,https://www.aldiwan.net/poem16184.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,None,None,None
3,لو كنت أطمع بالمنام توهما,بحر الكامل,"[لَو كُنتَ أَطمَع بِالمَنام تَوهما, لَسالَت طَ...",قصيدة عامه,https://www.aldiwan.net/poem16185.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,None,None,None
4,يعد علي أنفاسي ذنوبا,بحر الوافر,"[يعد عَليَّ أَنفاسي ذُنوباً, إِذا ما قُلت أَفد...",قصيدة عامه,https://www.aldiwan.net/poem16186.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,None,None,None


In [17]:
len(str(df["poem verses"].head(123)))  # length of the poem in the 123rd record

672

In [35]:
# Expand poems into bayts (pair lines)
rows = []

for _, row in df.iterrows():
    verses = row["poem verses"]
    meter = row["poem meter"]

    # pair every two lines as one bayt
    for i in range(0, len(verses)-1, 2):
        sadr = verses[i].strip()
        ajz = verses[i+1].strip()
        full_bayt = sadr + " # " + ajz
        rows.append({"verse": full_bayt, "meter": meter})

expanded_df = pd.DataFrame(rows)


In [36]:
print(rows[10])

{'verse': 'مِن أَي مَولى اِرتَجي # وَلاي باب التَجي', 'meter': 'بحر مجزوء الرمل'}


In [37]:
expanded_df.shape # shape of verse by verse dataset

(3845240, 2)

In [38]:
expanded_df["meter"] = expanded_df["meter"].fillna("Unknown_Meter")
# Count occurrences

meter_counts = expanded_df["meter"].value_counts()
meters_to_keep = meter_counts[meter_counts >= 100].index
filtered_df = expanded_df[expanded_df["meter"].isin(meters_to_keep)]

In [39]:
set(filtered_df["meter"].values)

{'Unknown_Meter',
 'البسيط',
 'التفعيله',
 'الخفيف',
 'الدوبيت',
 'الرجز',
 'الرمل',
 'السريع',
 'السلسلة',
 'الطويل',
 'الكامل',
 'المتدارك',
 'المتقارب',
 'المجتث',
 'المديد',
 'المسحوب',
 'المقتضب',
 'المنسرح',
 'المواليا',
 'الموشح',
 'الهزج',
 'الوافر',
 'بحر أحذ الكامل',
 'بحر البسيط',
 'بحر التفعيله',
 'بحر الخفيف',
 'بحر الدوبيت',
 'بحر الرجز',
 'بحر الرمل',
 'بحر السريع',
 'بحر الطويل',
 'بحر الكامل',
 'بحر المتدارك',
 'بحر المتقارب',
 'بحر المجتث',
 'بحر المديد',
 'بحر المقتضب',
 'بحر المنسرح',
 'بحر المواليا',
 'بحر الهزج',
 'بحر الوافر',
 'بحر مجزوء البسيط',
 'بحر مجزوء الخفيف',
 'بحر مجزوء الرجز',
 'بحر مجزوء الرمل',
 'بحر مجزوء السريع',
 'بحر مجزوء الكامل',
 'بحر مجزوء المتدارك',
 'بحر مجزوء المتقارب',
 'بحر مجزوء الوافر',
 'بحر مجزوء موشح',
 'بحر مخلع البسيط',
 'بحر مشطور الرجز',
 'بحر منهوك الرجز',
 'بحر منهوك المنسرح',
 'بحر موشح',
 'شعر التفعيلة',
 'شعر حر',
 'عامي',
 'عموديه',
 'نثريه'}

In [40]:
# to fix the inconsistency
mapping = {
    'بحر السريع':"السريع",
    "السريع":"السريع",
    "بحر الطويل": "الطويل",
    "الطويل": "الطويل",
    "بحر المديد": "المديد",
    "المديد": "المديد",
    "بحر البسيط": "البسيط",
    "البسيط": "البسيط",
    "بحر الكامل": "الكامل",
    "الكامل": "الكامل",
    "بحر الخفيف": "الخفيف",
    "الخفيف": "الخفيف",
    "بحر الوافر": "الوافر",
    "الوافر": "الوافر",
    "بحر الرجز": "الرجز",
    "الرجز": "الرجز",
    "بحر الرمل": "الرمل",
    "الرمل": "الرمل",
    "بحر المتدارك": "المتدارك",
    "المتدارك": "المتدارك",
    "بحر المتقارب": "المتقارب",
    "المتقارب": "المتقارب",
    "بحر المجتث": "المجتث",
    "المجتث": "المجتث",
    "بحر المقتضب": "المقتضب",
    "المقتضب": "المقتضب",
    "بحر المنسرح": "المنسرح",
    "المنسرح": "المنسرح",
    "بحر الموشح": "الموشح",
    "الموشح": "الموشح",
    "بحر الهزج": "الهزج",
    "الهزج": "الهزج",
    "المسحوب": "المضارع",  # variant mapping
}

In [41]:
filtered_df["meter"] = filtered_df["meter"].map(mapping)

/tmp/ipykernel_4062/3492382029.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df["meter"] = filtered_df["meter"].map(mapping)


In [42]:
canonical_meters = list(set(mapping.values()))
filtered_df = filtered_df[filtered_df["meter"].isin(canonical_meters)]

In [43]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

In [44]:
filtered_df["meter"].value_counts()

,count
meter,
الطويل,431097
الكامل,357313
البسيط,247383
الخفيف,165008
الوافر,144680
الرجز,113056
الرمل,70624
المتقارب,68358
السريع,60703


In [45]:
print(filtered_df.shape)
print("\n ----------------")
s=filtered_df.sample(1)
print(s)

(1747883, 2)

 ----------------
                                                                     verse  \
213592  سُلطانِ أَرضِ اللَهِ الَّذي ضَمِنَت # رِماحُهُ نَصرَ كُلِّ مَحروبِ   

          meter  
213592  المنسرح  


In [46]:
filtered_df["verse"].head(20)

,verse
0,أَصبَحَ المُلك لِلَّذي فَطر الخَل # قَ بِتَقديرٍ للعَزيز العَليمِ
1,غافر الذَنب للمسيءِ بِعَفوٍ # قابل التَوب ذي العَطاء العَميمِ
2,مُرسل المُصطَفى البَشير إِلَينا # رَحمة مِنهُ بِالكَلام القَديمِ
3,رَبَنا رَبّنا إِلَيكَ أَنينا # فَأَجرنا مِن حَر نار الجَحيمِ
4,وَاكفِنا شَرّ ما نَخاف بِلُطفٍ # يا عَظيماً يَرجى لِكُل عَظيمِ
5,وَتَقبل أَعمالَنا وَاعفُ عَنا # وَأَنلنا دُخول دار النَعيمِ
6,بِنَبي بَعثَتهُ فَهَدانا # لِصِراط مِن الهُدى مُستَقيمِ
7,وَبِمَن نَحنُ في حِماهُ مَدى الدَهر # أَخيهِ يَحيى الحصور الكَريمِ
8,أَدرك أَدرك قَوماً أَتوا بافتقار # وَاِنكِسار وَمَدمَع مَسجومِ
9,شَهدت أَرواحَهُم أَنكَ اللَهُ # وَجاءوا بِكُل قَلبٍ سَليم


In [53]:

import re

# Before cleaning - check current state
print("📊 BEFORE removing elongated verses:")
print(f"Total verses: {len(filtered_df):,}")
print("\nMeter distribution:")
print(filtered_df['meter'].value_counts())

# Function to detect repeated elongation marks
def has_repeated_elongation(text):
    """
    Detect if text contains repeated Arabic elongation marks (tatweel/kashida)
    Returns True if found, False otherwise
    """
    # Pattern for 2+ consecutive elongation marks
    pattern = r'ـ{2,}'
    return bool(re.search(pattern, str(text)))

# Identify verses with elongation marks
elongated_mask = filtered_df['verse'].apply(has_repeated_elongation)

print(f"\n🔍 Found {elongated_mask.sum():,} verses with repeated elongation marks")
print("\nElongated verses by meter:")
elongated_by_meter = filtered_df[elongated_mask]['meter'].value_counts()
print(elongated_by_meter)

# Remove elongated verses
filtered_df_clean = filtered_df[~elongated_mask].copy()

# Reset index
filtered_df_clean = filtered_df_clean.reset_index(drop=True)

print(f"\n✅ AFTER removing elongated verses:")
print(f"Total verses: {len(filtered_df_clean):,}")
print(f"Removed: {len(filtered_df) - len(filtered_df_clean):,} verses")

print("\nFinal meter distribution:")
final_counts = filtered_df_clean['meter'].value_counts()
print(final_counts)

📊 BEFORE removing elongated verses:
Total verses: 1,747,883

Meter distribution:
meter
الطويل      431097
الكامل      357313
البسيط      247383
الخفيف      165008
الوافر      144680
الرجز       113056
الرمل        70624
المتقارب     68358
السريع       60703
المنسرح      29364
المجتث       18789
الموشح       18160
الهزج         8360
المديد        7586
المتدارك      5470
المضارع       1131
المقتضب        801
Name: count, dtype: int64

🔍 Found 815,211 verses with repeated elongation marks

Elongated verses by meter:
meter
الطويل      185461
الكامل      175162
البسيط      108354
الخفيف       80472
الرجز        73034
الوافر       53033
الرمل        39726
المتقارب     29607
السريع       24516
الموشح       15665
المنسرح      10437
المجتث        9233
الهزج         3474
المتدارك      2997
المديد        2935
المضارع        782
المقتضب        323
Name: count, dtype: int64

✅ AFTER removing elongated verses:
Total verses: 932,672
Removed: 815,211 verses

Final meter distribution:
meter
الطويل     

In [ ]:
import pandas as pd

df = filtered_df_clean.copy()

# meters i want to keep completely
keep_all = ['المجتث', 'المنسرح']

# meters with more than 30000 records
meter_counts = df['meter'].value_counts()
big_meters = meter_counts[meter_counts > 30000].index.tolist()

final_parts = []

# process big meters
for meter in big_meters:

    subset = df[df['meter'] == meter]

    # remove duplicate verses
    subset = subset.drop_duplicates(subset='verse')

    # take 20000 samples so we have a balanced dataa
    subset = subset.sample(n=min(20000, len(subset)), random_state=42)

    final_parts.append(subset)

#keep all records of المجتث و المنسرح
small_subset = df[df['meter'].isin(keep_all)]
final_parts.append(small_subset)

#combine everything
final_df = pd.concat(final_parts).reset_index(drop=True)

print("Final meter distribution:")
print(final_df['meter'].value_counts())

Final meter distribution:
meter
الطويل      20000
الكامل      20000
البسيط      20000
الوافر      20000
الخفيف      20000
الرجز       20000
المتقارب    20000
السريع      20000
الرمل       20000
المنسرح     18927
المجتث       9556
Name: count, dtype: int64


In [56]:
final_df.shape

(208483, 2)

In [57]:
final_df.head(123)

,verse,meter
0,وَعَوّلَ في العسرِ الفَقيرُ على نَدى # يَدَيكَ وَهَل يَغْنى الكسيرُ عَنِ الجبرِ,الطويل
1,يـدمـن هـواهـا واللعـيـن وزيـرهـا # اغـثـنـي وادركـنـي بـنفسك والجيش,الطويل
2,إذا رُمْتُ نُصحاً جئت بالنصح واضحاً # وما كان من شأني الكلام المُعَقَّد,الطويل
3,يَبكي عَلى وَسنَةٍ تَزَوَّدَها # جيرانُهُ بَل بَكى مِنَ السَهَدِ,الطويل
4,إِذا ما أَقَمتُ الوَزنَ في نَظمِ وَصفِهِم # تَجودُ يَداهُم بِالنُضارِ بِلا وَزنِ,الطويل
5,لقـد خـوّفَـتـنـي بـالهَـلاكِ عـصـابَـةٌ # أُسـائِلُ عـنـهـا مـا تـقولُ اللهاذِمُ,الطويل
6,وخلي وداعي لا لصرح أقمته # ولكن لروح فيك قد جاوز الصرحا,الطويل
7,وهل تسمع الصّمُّ الدُّعاء وتُرتجى # موارد لا تفنى من الآل لُمَّعا,الطويل
8,وينثرُ هام الصيد والسيف مُغمدٌ # اذا ذلَّ في اِشْهارهِ الأمَراءُ,الطويل
9,فَـأَقْـوَتْ مِـنَ السّـاداتِ مِنْ آلِ هاشِم # وَلَمْ يَـجْـتَـمِـعْ بَعْدَ الْحُسَيْنِ شُتاتُها,الطويل


In [61]:
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Check first few rows
print(final_df.head())

                                                                       verse  \
0           وَمِن حَبيرِ اليُمنَةِ الأَبرادِ # مِن أَتحَمِيّاتٍ وَمِن وِرادِ   
1  أَبا العُلوانِ فَضلُكَ قالَ شِعرِي # وَلِلفَضلِ اِشتِهارُ أُولِي الفِضالِ   
2                     مـالي عـلى صومك من رشفة # فـاكِ وإن أظـمـاك مـن لَوْمِ   
3                             وما أنت إلا كباقي مناي # تموت وتولد بين الضلوع   
4                     قَـد كـانَ لِلمـالِ رَبّـاً # فَصارَ في البُخلِ عَبدَه   

      meter  
0     الرجز  
1    الوافر  
2    السريع  
3  المتقارب  
4    المجتث  


In [62]:
# splitting in the other notebook
final_df.to_csv("final_poem_dataset_short_letters.csv",index=False, encoding="utf-8-sig")

In [63]:
final_df["meter"].value_counts()

,count
meter,
الرجز,20000
الوافر,20000
السريع,20000
المتقارب,20000
الطويل,20000
الكامل,20000
البسيط,20000
الرمل,20000
الخفيف,20000
